# 04 - 类与面向对象（Java vs Python）

Java 开发者最熟悉 OOP，但 Python 的 class 有很多"反直觉"的地方。

## 今日目标（25-35 分钟）

- 掌握 `self`、`__init__`、访问控制的差异
- 会用魔术方法让自定义类支持 `print`、`len`、`==` 等操作
- 会用 `dataclass` 替代样板代码（对比 Java 的 record / Lombok）
- 了解多继承和 Protocol

完成标准：通过末尾打卡题

## 1. class 定义：`self` 对比 Java 的 `this`

In [ ]:
# 说明：演示类定义与初始化流程，重点观察实例属性与方法调用方式。

# Java:
# public class User {
#     private String name;
#     public User(String name) { this.name = name; }
#     public String getName() { return this.name; }
# }

class User:
    # __init__ 就是构造函数（Java 的 constructor）
    def __init__(self, name: str, age: int):
        self.name = name  # self 就是 Java 的 this，但必须显式写出来
        self.age = age

    def greet(self) -> str:
        # 每个实例方法的第一个参数都是 self（Java 隐式传 this）
        return f"Hi, I'm {self.name}, age {self.age}"

u = User("kai", 18)
print(u.greet())
print(u.name)  # 直接访问属性，没有 getter


In [ ]:
# 说明：演示 Python 对象属性可动态添加，以及这种灵活性的代价。

# 关键区别：Python 的属性在 __init__ 里动态创建，不需要提前声明
# Java 必须在类体里先声明 private String name;

class Flexible:
    def __init__(self):
        self.x = 10

f = Flexible()
f.y = 20  # 甚至可以在外部随时加属性！Java 不可能做到
print(f.x, f.y)

# ⚠️ 坑：动态加属性虽然灵活，但滥用会让代码难以维护
# 你不知道一个对象到底有哪些属性，IDE 也无法补全 __init__ 外声明的属性
# 生产代码里应该把所有属性都在 __init__ 里统一声明
# 或者用 @dataclass，让 Pylance 能做完整的静态分析


## 2. 访问控制：没有 private / protected / public

In [ ]:
# 说明：演示单下划线与双下划线命名约定，理解 Python 的“约定式封装”。

class Account:
    def __init__(self, owner: str, balance: float):
        self.owner = owner       # 公开属性（相当于 public）
        self._balance = balance  # 单下划线：约定私有，外部"不应该"访问，但实际能访问
        self.__secret = "abc"   # 双下划线：名称改写（name mangling），外部不能直接用 .__ 访问

a = Account("kai", 100.0)
print(a.owner)      # 正常访问
print(a._balance)   # 能访问，但 IDE 会提示你不该用
# print(a.__secret) # AttributeError!
print(a._Account__secret)  # 双下划线实际被改写成了 _类名__属性名，能绕过但别这么干

# 总结：Python 靠约定而不是强制，单下划线 _ 就够了

# ⚠️ 坑：双下划线 __ 有个继承陷阱——
# 子类里用 self.__secret 访问不到父类的 __secret（因为名称已被改写成 _Account__secret）
# 实际开发中：用单下划线 _ 表示"内部使用"就够了
# 双下划线只在需要防止子类意外覆盖同名属性的极少数场景才用


## 3. 继承

In [ ]:
# 说明：演示继承、super() 与方法重写，对照 Java extends + @Override。

# Java: class Admin extends User { ... }
# Python: class Admin(User):

class Admin(User):
    def __init__(self, name: str, age: int, role: str):
        super().__init__(name, age)  # 调用父类构造函数，和 Java 的 super() 一样
        self.role = role

    def greet(self) -> str:  # 方法重写，不需要 @Override
        return f"Hi, I'm {self.name} ({self.role})"

a = Admin("kai", 18, "superadmin")
print(a.greet())
print(isinstance(a, User))  # True，和 Java 的 instanceof 一样


In [ ]:
# 说明：演示多继承与 MRO，理解 Python 方法查找顺序。

# 多继承（Java 不支持，只能多实现接口）

class Loggable:
    def log(self, msg: str):
        print(f"[LOG] {msg}")

class Serializable:
    def to_dict(self) -> dict:
        return self.__dict__  # __dict__ 返回对象所有属性的 dict

class Service(Loggable, Serializable):
    def __init__(self, name: str):
        self.name = name

s = Service("api")
s.log("started")       # 来自 Loggable
print(s.to_dict())     # 来自 Serializable

# 多继承在 Python 里常见，但一般用 Mixin 模式（像上面的 Loggable），不会搞复杂的菱形继承

# ⚠️ 坑：多继承同名方法的查找顺序由 MRO（Method Resolution Order）决定
# Python 用 C3 线性化算法，可以用 .__mro__ 查看实际顺序：
print(Service.__mro__)
# 输出：(Service, Loggable, Serializable, object)
# 规则：括号里靠左的父类优先——class Service(Loggable, Serializable) 中 Loggable 先被搜索
# 如果两个父类有同名方法，左边的会胜出（静默覆盖，不报错）


## 4. 魔术方法（对比 Java 的 toString / equals / hashCode）

In [ ]:
# 说明：演示 __str__/__repr__/__eq__/__len__ 等魔术方法的行为接管。

class Point:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y

    def __str__(self) -> str:
        # print() 时调用，对比 Java 的 toString()
        return f"Point({self.x}, {self.y})"

    def __repr__(self) -> str:
        # 调试/日志时调用，建议返回能重建对象的表达式
        return f"Point({self.x}, {self.y})"

    def __eq__(self, other) -> bool:
        # == 运算符，对比 Java 的 equals()
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def __len__(self) -> int:
        # len() 调用时触发，Java 没有等价物
        return abs(self.x) + abs(self.y)

p1 = Point(3, 4)
p2 = Point(3, 4)
p3 = Point(1, 2)

print(p1)          # 调用 __str__
print(p1 == p2)    # True，调用 __eq__
print(p1 == p3)    # False
print(len(p1))     # 7，调用 __len__


In [ ]:
# 说明：演示魔术方法如何接管 print、==、len 等内置行为。

# 常用魔术方法速查
# __init__     构造函数
# __str__      print() 时的字符串表示（Java toString）
# __repr__     调试时的字符串表示
# __eq__       == 运算符（Java equals）
# __hash__     hash() 用于 dict/set（Java hashCode）
# __len__      len() 调用
# __getitem__  obj[key] 取值
# __setitem__  obj[key] = val 赋值
# __contains__ in 运算符
# __iter__     for 循环迭代
# __call__     obj() 把对象当函数调用

# 不需要背，用到查就行。知道有这个机制即可


## 5. dataclass（对比 Java 的 record / Lombok）

写普通 class 太多样板代码？`dataclass` 帮你自动生成 `__init__`、`__repr__`、`__eq__` 等。

In [ ]:
# 说明：演示 dataclass 自动生成构造、打印和比较逻辑，减少样板代码。

from dataclasses import dataclass

# Java:
# public record User(String name, int age) {}
# 或者用 Lombok: @Data public class User { ... }

@dataclass
class UserProfile:
    name: str
    age: int
    active: bool = True  # 默认值

u1 = UserProfile("kai", 18)
u2 = UserProfile("kai", 18)

print(u1)          # User(name='kai', age=18, active=True)  自动生成 __repr__
print(u1 == u2)    # True  自动生成 __eq__
print(u1.name)     # 直接访问


In [ ]:
# 说明：演示 dataclass 自动生成构造、打印和比较逻辑，减少样板代码。

from dataclasses import dataclass, field

@dataclass
class Config:
    host: str = "localhost"
    port: int = 8000
    tags: list = field(default_factory=list)  # 可变类型的默认值要用 field

c1 = Config()
c2 = Config(host="0.0.0.0", port=3000)

print(c1)
print(c2)

# 和 03 里的可变默认参数坑一样的道理：
# tags: list = []  会导致所有实例共享同一个 list
# tags: list = field(default_factory=list)  每次创建新 list

# 💡 实战：AI 应用里推荐用 pydantic 的 BaseModel 代替 dataclass
# pydantic 会在运行时自动做类型验证，dataclass 只是结构糖、不验证类型：
#
# from pydantic import BaseModel
# class Config(BaseModel):
#     host: str = "localhost"
#     port: int = 8000
#
# Config(port="abc")  # pydantic 会报 ValidationError，dataclass 不会
# FastAPI 的请求体 / Langchain 的配置项都大量使用 pydantic BaseModel


In [ ]:
# 说明：演示 dataclass 自动生成构造、打印和比较逻辑，减少样板代码。

# frozen=True 让 dataclass 不可变（类似 Java 的 record）

@dataclass(frozen=True)
class Coordinate:
    lat: float
    lng: float

c = Coordinate(31.23, 121.47)
print(c)

try:
    c.lat = 0  # 报错！frozen 不允许修改
except Exception as e:
    print(f"Error: {e}")


## 6. 没有 interface，用抽象类或 Protocol

In [ ]:
# 说明：演示抽象基类（ABC）定义接口契约并强制子类实现。

# 方式一：抽象类（对比 Java 的 abstract class / interface）
from abc import ABC, abstractmethod

class Repository(ABC):
    @abstractmethod
    def find_by_id(self, id: int) -> dict:
        pass  # pass 表示空方法体

    @abstractmethod
    def save(self, data: dict) -> None:
        pass

# 不能直接实例化
try:
    r = Repository()
except TypeError as e:
    print(f"Error: {e}")

# 必须实现所有抽象方法
class UserRepository(Repository):
    def find_by_id(self, id: int) -> dict:
        return {"id": id, "name": "kai"}

    def save(self, data: dict) -> None:
        print(f"saved: {data}")

repo = UserRepository()
print(repo.find_by_id(1))
repo.save({"name": "kai"})


In [ ]:
# 说明：本段示例对应「6. 没有 interface，用抽象类或 Protocol」，建议先看注释再运行，重点观察输出与预期是否一致。

# 方式二：Protocol（Python 3.8+，结构化类型，类似 TypeScript 的 interface）
# 不需要显式继承，只要有对应的方法就算实现了
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> str: ...

class Circle:
    def draw(self) -> str:  # 没有继承 Drawable，但有 draw 方法
        return "drawing circle"

class Square:
    def draw(self) -> str:
        return "drawing square"

# 类型检查器（Pylance）会认为 Circle 和 Square 满足 Drawable
# 这叫 "鸭子类型"：不看你是什么类，看你有没有这个方法
def render(shape: Drawable) -> None:
    print(shape.draw())

render(Circle())
render(Square())


## 7. Java 对照速记

| Java | Python |
|------|--------|
| `this` | `self`（必须显式写在参数里） |
| 构造函数 `ClassName()` | `__init__(self)` |
| `private` | `_` 前缀（约定，不强制） |
| `extends` | `class Child(Parent):` |
| `toString()` | `__str__()` |
| `equals()` | `__eq__()` |
| `hashCode()` | `__hash__()` |
| `record` / Lombok `@Data` | `@dataclass` |
| `interface` | `ABC` + `@abstractmethod` 或 `Protocol` |
| 单继承 + 多接口 | 多继承（Mixin 模式） |

## 今日打卡题

1. 用 `dataclass` 定义一个 `Task`，包含 `title: str`、`done: bool = False`，打印它并比较两个相同的 Task 是否相等
2. 写一个 `Animal` 基类（带 `speak` 抽象方法），`Dog` 和 `Cat` 分别实现，`Dog.speak()` 返回 `"woof"`，`Cat.speak()` 返回 `"meow"`
3. 给下面的 `Counter` 类补上魔术方法，让它支持 `print()`、`len()`：

```python
class Counter:
    def __init__(self, items: list):
        self.items = items
```

In [ ]:
# 说明：这是练习留空区域，建议先独立完成，再运行后续参考答案进行对照。

# TODO: 先自己写


In [ ]:
# 说明：这是打卡题参考实现，建议先自己做一遍，再用它核对思路和语法。

# 参考答案（先自己写完再看）

# 1
from dataclasses import dataclass

@dataclass
class Task:
    title: str
    done: bool = False

t1 = Task("learn python")
t2 = Task("learn python")
print(t1)          # Task(title='learn python', done=False)
print(t1 == t2)    # True

# 2
from abc import ABC, abstractmethod

class Animal(ABC):
    @abstractmethod
    def speak(self) -> str:
        pass

class Dog(Animal):
    def speak(self) -> str:
        return "woof"

class Cat(Animal):
    def speak(self) -> str:
        return "meow"

print(Dog().speak())  # woof
print(Cat().speak())  # meow

# 3
class Counter:
    def __init__(self, items: list):
        self.items = items

    def __str__(self) -> str:
        return f"Counter({self.items})"

    def __len__(self) -> int:
        return len(self.items)

c = Counter([1, 2, 3])
print(c)       # Counter([1, 2, 3])
print(len(c))  # 3


下一步建议：进入 `05-exception-file.ipynb`，掌握 `try/except` 和 `with` 语句。